In [1]:
# Add module to path
import os
import sys
from itertools import product
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colors
import pandas as pd
from tqdm import tqdm
from itertools import product
from numba import jit, njit, types
from numba.typed import Dict
import time

# from darts.dartboards import generate_dartboard
from darts.mdp import SinglePlayerContinuousMDP
from darts.stats import expected_score

In [3]:
from darts.dartboards import DARTBOARD_CONSTANTS

In [ ]:
board_size = 128
sigma = 10

mm_per_pixel = 2*DARTBOARD_CONSTANTS['DARTBOARD_RADIUS_MM']/board_size
sigma_pxl = sigma / mm_per_pixel

Sigma = sigma_pxl*sigma_pxl*np.array([[1, 0], [0, 1]])
margin = 0.25*sigma_pxl
game_start = 501

In [5]:
mdp = SinglePlayerContinuousMDP(board_size, Sigma, margin, game_start, point_stride=1)

In [6]:
_ = mdp.probs

  0%|          | 0/7441 [00:00<?, ?it/s]

  0%|          | 1/7441 [00:01<2:42:45,  1.31s/it]

  4%|▍         | 316/7441 [00:01<00:22, 309.82it/s]

  9%|▊         | 638/7441 [00:01<00:10, 667.46it/s]

 13%|█▎        | 956/7441 [00:01<00:06, 1042.64it/s]

...

 86%|████████▋ | 6436/7441 [00:03<00:00, 3218.07it/s]

 91%|█████████ | 6763/7441 [00:03<00:00, 3230.96it/s]

 95%|█████████▌| 7087/7441 [00:03<00:00, 3221.98it/s]

100%|█████████▉| 7410/7441 [00:03<00:00, 3193.95it/s]

100%|██████████| 7441/7441 [00:03<00:00, 2048.62it/s]

In [ ]:
from darts.mdp_3turn import build_probs_arrays, compute_3turn_values, _compute_3turn_values

# Convert probs once — this is shared across all game_start values
probs_arr, checkout_probs_arr, allowed_scores = build_probs_arrays(mdp)
print(f"n_points: {probs_arr.shape[0]},  n_scores: {probs_arr.shape[1]}")
print(f"allowed_scores: {allowed_scores}")

In [ ]:
import time

# --- Warm up Numba JIT (compiles the function, not counted in timings) ---
_v_warmup = np.zeros((7, 3, 7), dtype=np.float64)
_compute_3turn_values(_v_warmup, probs_arr, checkout_probs_arr, allowed_scores, 5, 0.1)
print("JIT compilation done")

# --- Profile: time at several game_start values and fit a quadratic ---
profile_game_starts = [20, 50, 100, 200]
times = []

for gs in profile_game_starts:
    v = np.zeros((gs + 2, 3, gs + 2), dtype=np.float64)
    t0 = time.perf_counter()
    _compute_3turn_values(v, probs_arr, checkout_probs_arr, allowed_scores, gs, 1e-4)
    elapsed = time.perf_counter() - t0
    times.append(elapsed)
    print(f"  game_start={gs:>3}: {elapsed:6.2f}s | "
          f"V(2,1,2)={-v[2,0,2]:.2f} darts | V({gs},1,{gs})={-v[gs,0,gs]:.2f} darts")

# Fit time ~ a*n^2 + b*n + c (theory: complexity is quadratic in game_start)
xs = np.array(profile_game_starts, dtype=float)
ys = np.array(times)
coeffs = np.polyfit(xs, ys, 2)
a, b, c = coeffs

t501 = a * 501**2 + b * 501 + c
print(f"\nQuadratic fit:  t ≈ {a:.2e}·n² + {b:.2e}·n + {c:.2e}")
print(f"Projected time for game_start=501: {t501:.0f}s  ({t501/60:.1f} min)")

In [ ]:
# --- Full run: game_start = 501 ---
t0 = time.perf_counter()
values = compute_3turn_values(mdp, threshold=1e-4)
elapsed = time.perf_counter() - t0
print(f"game_start=501 completed in {elapsed:.1f}s ({elapsed/60:.1f} min)")

# Key results: expected darts to checkout from each score at the start of a fresh round
print("\nExpected darts to checkout (turn 1, fresh round):")
for score in [501, 400, 300, 200, 100, 50, 40, 32, 20, 10, 4, 2]:
    print(f"  score={score:>3}: {-values[score, 0, score]:.2f} darts")

In [ ]:
# --- Cross-check: compare 3-turn vs 1-turn MDP values ---
# The published single-player MDP (darts/mdp.py) has no round structure: a bust
# instantly resets to the same score. The 3-turn model should always give slightly
# higher expected dart counts because busts mid-round forfeit remaining darts.
# We compute 1-turn values directly via the jitted _compute_state_value to avoid
# running _compute_actions (policy extraction, not needed for the comparison).
from numba.typed import Dict
from numba import types
from darts.mdp import _compute_state_value

# Build 1-turn value dict and run state-by-state iteration (mirrors mdp.compute_values)
d_values_1t = Dict.empty(key_type=types.int32, value_type=types.float64)
for k in range(game_start + 1):
    d_values_1t[np.int32(k)] = 0.0

d_probs_1t = Dict.empty(key_type=types.string, value_type=types.DictType(types.int32, types.float64))
for k, v in mdp.probs["probs"].items():
    d_probs_1t[",".join(str(x) for x in k)] = v

d_cprobs_1t = Dict.empty(key_type=types.string, value_type=types.DictType(types.int32, types.float64))
for k, v in mdp.probs["checkout_probs"].items():
    d_cprobs_1t[",".join(str(x) for x in k)] = v

print("Running 1-turn value iteration...")
for state in range(game_start + 1):
    tmp = Dict.empty(key_type=types.int32, value_type=types.float64)
    for k, v in d_values_1t.items():
        tmp[k] = v
    if state >= 2:
        tmp[np.int32(state)] = tmp[np.int32(state - 1)]
    d_values_1t = _compute_state_value(np.int32(state), tmp, mdp.points, d_probs_1t, d_cprobs_1t, 1e-4)
print("Done")

print("\nScore | 1-turn MDP | 3-turn MDP | Difference")
print("------+------------+------------+-----------")
for score in [501, 300, 100, 50, 40, 32, 20, 10, 4, 2]:
    v1 = -float(d_values_1t[np.int32(score)])
    v3 = -values[score, 0, score]
    print(f"  {score:>3} | {v1:>10.2f} | {v3:>10.2f} | {v3-v1:>+.3f}")